# 22. Robustness check: look-back window of the trailing peak in the crash definition

I define crashes as a 4% fall from the trailing four-hour high; here I ask whether the conclusions change when that look-back window H changes. I use Binance spot BTCUSDT one-minute data over the development period 2023-01 to 2026-04.

- I vary only H = 120, 240 (reference), 480 and 1440 minutes. The peak is the highest close in the H minutes before the breach and the onset is the first close 1% below that peak. I build the events with `price_only_catalog`, which implements the notebook 02 rule, and change only `H` (5-minute sustain, 24-hour refractory period and new-peak rule unchanged).
- I keep fixed the 4% decline, the judging window (30–240 minutes before the breach) and the models, scores and alerts frozen in notebook 16 (four methods), which I do not retrain. I ask whether the accounting changes with the evaluation definition, not whether a rebuilt model does better.
- For evaluation I use the per-minute AUC decomposition of notebooks 18 and 21 with a 7-day block bootstrap, and alert-level counts against three nulls (circular shift of at least 7 days, short shift of 4 hours to 1 day, regime-matched).

I fixed the reading rules before seeing results and recorded them in `22_SPEC.json`. I call the conclusion that in-progress detection is a substantial part of standard skill robust to H if the lower 95% bound of M0's in-progress share of the above-chance AUC is at least 0.25 at every H; otherwise I report the H at which it fails.

This analysis is exploratory. I have already examined the development period repeatedly, so I treat these results as robustness checks only, and I do not use the holdouts. I write the results to `btc_lead_cost_results_77months_spot/peak_window_v1/`. The last cells of this saved copy have no output; `PEAK_WINDOW_SUMMARY.json` holds the results and the reading.

In [1]:
from pathlib import Path
from collections import deque, defaultdict
import io, os, sys, json, math, time, hashlib, zipfile, warnings, urllib.request, urllib.error
import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.stats import binomtest
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
try:
    import pyarrow  # noqa: F401
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'pyarrow'])
try:
    from numba import njit
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'numba'])
    from numba import njit
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Colab: paths below must point to a folder with the same structure.')

DRIVE = Path('/content/drive/MyDrive')
SPOT_BASE = DRIVE / 'btc_lead_cost_results_77months_spot'      # spot results (notebooks 01–07)
FUT_BASE = DRIVE / 'btc_lead_cost_results_77months'            # futures results, read only
BINANCE = 'https://data.binance.vision/data/'
NS = 60 * 10**9
LEAD_MIN = 30; LEAD_MAX = 240; COOLDOWN = 240; COOLDOWN_MIN = 240
B = 240; MONTH_CAP = 20
EVAL_START = '2023-01-01'; EVAL_END = '2026-05-01'

_src = SPOT_BASE / 'cache' / 'DATA_SOURCE.json'
if not _src.is_file() or json.loads(_src.read_text()).get('venue') != 'spot':
    raise RuntimeError(f'Spot cache not found: {_src}. Run notebook 01 first.')
REF_PATH = SPOT_BASE / 'causal_alert_selection_v1' / 'spot_reference.json'
THR_PATH = SPOT_BASE / 'FINAL_risk_adaptive_alert_capacity_v1' / 'spot_frozen_thresholds.json'
for _p in (REF_PATH, THR_PATH):
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 05 and 06 are required: {_p}')
REF = json.loads(REF_PATH.read_text())
FROZEN = json.loads(THR_PATH.read_text())
assert REF.get('venue') == 'spot' and FROZEN.get('venue') == 'spot'
TH_CAL = float(REF['TH_CAL']); TH_HIGH = float(REF['TH_HIGH']); TH_BUCKET = float(FROZEN['TH_BUCKET'])
print('SPOT cache verified. Frozen spot thresholds:', dict(TH_CAL=TH_CAL, TH_HIGH=TH_HIGH, TH_BUCKET=TH_BUCKET))


def sha256_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def load_btc_spot():
    px = pd.read_parquet(SPOT_BASE / 'cache' / 'BTCUSDT_1m_regularized.parquet').sort_index()
    sg = pd.read_parquet(SPOT_BASE / 'cache' / 'btc_signals_1m.parquet', columns=['z_vol', 'tsr', 'z_hs']).sort_index()
    for fr in (px, sg):
        assert isinstance(fr.index, pd.DatetimeIndex) and fr.index.tz is not None
        fr.index = fr.index.tz_convert('UTC')
    assert px.index.equals(sg.index), 'price/signal caches must share one 1-minute index'
    assert px.index.is_unique and (np.diff(px.index.asi8) == NS).all()
    return px, sg


def load_catalog(path):
    E = pd.read_csv(path)
    for c in ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']:
        E[c] = pd.to_datetime(E[c], utc=True)
    E = E.sort_values('breach_utc').reset_index(drop=True)
    assert E.event_id.is_unique
    return E


def label_from_catalog(T, E):
    """1 if a 4% breach occurs 30-240 minutes after minute t (same rule as notebooks 04–07)."""
    b = E.breach_utc.astype('int64').to_numpy()
    pos = np.searchsorted(b, T + LEAD_MIN * NS, 'left')
    y = np.zeros(len(T), dtype=np.int8)
    ok = pos < len(b)
    y[ok] = (b[pos[ok]] <= T[ok] + LEAD_MAX * NS).astype(np.int8)
    return y


def ts(s):
    return pd.Timestamp(s, tz='UTC')


def in_period(T, start, end):
    return (T >= ts(start).value) & (T < ts(end).value)


def train_mask(T, valid, year=2021):
    return valid & (T >= ts(f'{year}-01-01').value) & (T < ts(f'{year + 1}-01-01').value - (LEAD_MAX + 5) * NS)


def new_logit():
    return make_pipeline(StandardScaler(), LogisticRegression(C=0.2, class_weight='balanced', max_iter=400,
                                                              solver='lbfgs', random_state=17))


def events_between(E, start, end):
    """Evaluation events: breach in [start+4h, end) so every warning window lies inside the period."""
    return E[(E.breach_utc >= ts(start) + pd.Timedelta(minutes=LEAD_MAX)) & (E.breach_utc < ts(end))].reset_index(drop=True)


# --- emit_calendar, as in notebook 05 ---
def emit_calendar(t,eligible,B=240,cap=20,cooldown_min=240):
    t=np.asarray(t,dtype=np.int64)
    eligible=np.asarray(eligible,dtype=bool)
    assert len(t)==len(eligible) and np.all(np.diff(t)>0)
    ids=np.flatnonzero(eligible)
    z=[]; mcount=0; ycount=0; pm=None; py=None; last=-(10**20)
    for k in ids:
        ti=int(t[k]); dt=pd.Timestamp(ti,tz='UTC')
        year=int(dt.year); month=year*12+int(dt.month)
        if year!=py: ycount=0; py=year
        if month!=pm: mcount=0; pm=month
        if ycount>=B or mcount>=cap or ti-last<cooldown_min*NS:continue
        z.append(ti); last=ti; ycount+=1; mcount+=1
    return np.asarray(z,dtype=np.int64)

# --- match_events, as in notebook 05 ---
def match_events(events,alerts):
    alerts=np.sort(np.asarray(alerts,dtype=np.int64))
    out=[];used=set()
    for ev in events.itertuples(index=False):
        breach=ev.breach_utc.value; onset=ev.onset_1pct_utc.value
        left=int(np.searchsorted(alerts,breach-LEAD_MAX*NS,'left'))
        right=int(np.searchsorted(alerts,breach-LEAD_MIN*NS,'right'))
        for j in range(left,right):
            if j not in used:
                used.add(j)
                out.append(dict(event_id=int(ev.event_id),breach_utc=ev.breach_utc,
                                alarm_utc=pd.Timestamp(int(alerts[j]),tz='UTC'),
                                lead_min=(breach-int(alerts[j]))/NS,
                                pre1pct=bool(alerts[j]<onset)))
                break
    return pd.DataFrame(out,columns=['event_id','breach_utc','alarm_utc','lead_min','pre1pct'])

# --- statistics, as in notebook 05 ---
def statistics(events,alerts):
    hits=match_events(events,alerts)
    n=len(alerts);N=len(events)
    return dict(alarms=n,caught=len(hits),before_1pct=int(hits.pre1pct.sum()),
                false_alarms=n-len(hits),recall=len(hits)/N,
                pre_onset_recall=int(hits.pre1pct.sum())/N,
                alert_precision=(len(hits)/n if n else 0))



def cal_fields(t):
    idx = pd.to_datetime(np.asarray(t, dtype='int64'), utc=True)
    years = idx.year.to_numpy().astype(np.int64)
    months = (idx.year.to_numpy() * 12 + idx.month.to_numpy()).astype(np.int64)
    return years, months


@njit(cache=False)
def _calendar_core(t, elig, years, months, B, caps, cooldown_ns):
    # Line-by-line port of emit_calendar with a per-minute monthly cap (caps) so a surge policy can reuse it.
    out = np.empty(len(t), np.int64); k = 0
    mcount = 0; ycount = 0; pm = -1; py = -1; last = -(2**62)
    for i in range(len(t)):
        if not elig[i]:
            continue
        ti = t[i]; yr = years[i]; mo = months[i]
        if yr != py:
            ycount = 0; py = yr
        if mo != pm:
            mcount = 0; pm = mo
        if ycount >= B or mcount >= caps[i] or ti - last < cooldown_ns:
            continue
        out[k] = ti; k += 1; last = ti; ycount += 1; mcount += 1
    return out[:k].copy()


_CAL_CACHE = [None, None]


def emit_calendar_fast(t, eligible, B=240, cap=20, cooldown_min=240, caps=None):
    t = np.asarray(t, dtype=np.int64); eligible = np.asarray(eligible, dtype=bool)
    assert len(t) == len(eligible) and (len(t) < 2 or np.all(np.diff(t) > 0))
    if _CAL_CACHE[0] is t:
        years, months = _CAL_CACHE[1]
    else:
        years, months = cal_fields(t); _CAL_CACHE[0] = t; _CAL_CACHE[1] = (years, months)
    if caps is None:
        caps = np.full(len(t), int(cap), np.int64)
    return _calendar_core(t, eligible, years, months, int(B), np.asarray(caps, np.int64), np.int64(cooldown_min * NS))


def check_fast_calendar(t, s):
    """Stop if the compiled calendar ever differs from the notebook 05 reference implementation."""
    qs = np.unique(np.quantile(s[np.isfinite(s)], [0., .5, .8, .9, .95, .99, .999]))
    for th in qs:
        e = np.isfinite(s) & (s >= th)
        for B_, cap_ in [(240, 20), (60, 5)]:
            if not np.array_equal(emit_calendar(t, e, B=B_, cap=cap_), emit_calendar_fast(t, e, B=B_, cap=cap_)):
                raise AssertionError(f'fast calendar differs from the reference at th={th}, B={B_}')
    return len(qs) * 2


def score_grid(sc):
    Q = np.unique(np.r_[0., np.linspace(.10, .995, 33), .997, .999, .9999, 1.])
    return np.unique(np.r_[np.nextafter(sc.min(), -np.inf), np.quantile(sc, Q), np.nextafter(sc.max(), np.inf)])


def calibrate_calendar(t_cal, s_cal, grid=None, B_=240, cap_=20):
    """Notebook 05 rule: the most 2022 alerts reachable (cap = B), then the fewest raw candidates (strictest)."""
    grid = score_grid(s_cal) if grid is None else grid
    rows = []
    for th in grid:
        e = s_cal >= th
        rows.append((float(th), len(emit_calendar_fast(t_cal, e, B=B_, cap=cap_)), int(e.sum())))
    df = pd.DataFrame(rows, columns=['threshold', 'cal_alerts', 'raw_candidates'])
    best = df[df.cal_alerts == df.cal_alerts.max()].sort_values('raw_candidates').iloc[0]
    return float(best.threshold), int(best.cal_alerts), df


def mcnemar_one_sided(a_only, b_only):
    """P(at least b_only successes | n discordant, p=.5): evidence that B beats A."""
    n = a_only + b_only
    return float(binomtest(b_only, n, .5, alternative='greater').pvalue) if n else 1.0


def mcnemar_two_sided(a_only, b_only):
    n = a_only + b_only
    return float(binomtest(min(a_only, b_only), n, .5).pvalue) if n else 1.0


def paired(events, base_alerts, new_alerts):
    mb = match_events(events, base_alerts); mn = match_events(events, new_alerts)
    hb, hn = set(mb.event_id), set(mn.event_id)
    pb, pn = set(mb.loc[mb.pre1pct, 'event_id']), set(mn.loc[mn.pre1pct, 'event_id'])
    return dict(base_caught=len(hb), new_caught=len(hn), base_only_caught=len(hb - hn), new_only_caught=len(hn - hb),
                p_caught_two_sided=mcnemar_two_sided(len(hb - hn), len(hn - hb)),
                base_pre1pct=len(pb), new_pre1pct=len(pn), base_only_pre1pct=len(pb - pn), new_only_pre1pct=len(pn - pb),
                p_pre1pct_two_sided=mcnemar_two_sided(len(pb - pn), len(pn - pb)),
                p_pre1pct_one_sided_new_better=mcnemar_one_sided(len(pb - pn), len(pn - pb)))
RAW_COLS = ['open_time', 'open', 'high', 'low', 'close', 'volume', 'close_time', 'quote_volume',
            'trade_count', 'taker_buy_vol', 'taker_buy_quote', 'ignore']


def month_list(first, last):
    return [p.strftime('%Y-%m') for p in pd.period_range(first, last, freq='M')]


def _fetch(url, tries=5):
    last = None
    for k in range(tries):
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=120) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            last = e
        except Exception as e:  # network hiccup: retry
            last = e
        time.sleep(2 ** k)
    raise RuntimeError(f'download failed: {url} ({last})')


def get_archives(prefix, symbol, months, dest, allow_missing=False):
    """Binance public monthly 1m klines (prefix 'spot' or 'futures/um'), SHA-256 checked against .CHECKSUM.
    Already-downloaded files whose hash matches are reused, so an interrupted run resumes."""
    dest = Path(dest); dest.mkdir(parents=True, exist_ok=True)
    rows = []
    for ym in months:
        url = f'{BINANCE}{prefix}/monthly/klines/{symbol}/1m/{symbol}-1m-{ym}.zip'
        z = dest / f'{symbol}-1m-{ym}.zip'
        chk = _fetch(url + '.CHECKSUM')
        if chk is None:
            if allow_missing:
                rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=None, status='not_listed'))
                continue
            raise RuntimeError(f'archive not published: {url}')
        expected = chk.decode().split()[0].strip().lower()
        if z.is_file() and sha256_file(z) == expected:
            status = 'cached'
        else:
            data = _fetch(url)
            if data is None:
                raise RuntimeError(f'archive missing although checksum exists: {url}')
            if hashlib.sha256(data).hexdigest() != expected:
                raise RuntimeError(f'SHA-256 mismatch: {url}')
            part = z.parent / (z.name + '.part'); part.write_bytes(data); part.replace(z)
            status = 'downloaded'
        rows.append(dict(symbol=symbol, venue=prefix, month=ym, url=url, sha256=expected, status=status))
    man = pd.DataFrame(rows)
    print(symbol, prefix, man.status.value_counts().to_dict(), flush=True)
    return man


def parse_kline_zip(path):
    """Header or no header; ms or microsecond timestamps (spot archives switched to microseconds in 2025)."""
    with zipfile.ZipFile(path) as zf:
        members = [m for m in zf.namelist() if m.lower().endswith('.csv')]
        if len(members) != 1:
            raise ValueError(f'expected one CSV in {path}')
        with zf.open(members[0]) as fd:
            x = pd.read_csv(fd, header=None, low_memory=False)
    if str(x.iloc[0, 0]).strip().lower() in ('open_time', 'timestamp'):
        x = x.iloc[1:].reset_index(drop=True)
    if x.shape[1] < 12:
        raise ValueError(f'unexpected kline schema in {path}: {x.shape[1]} columns')
    x = x.iloc[:, :12]; x.columns = RAW_COLS
    number = pd.to_numeric(x.open_time, errors='raise')
    mag = float(number.abs().median())
    unit = 'us' if mag > 1e14 else ('ms' if mag > 1e11 else 's')
    z = x[['open', 'high', 'low', 'close', 'volume', 'taker_buy_vol']].apply(pd.to_numeric, errors='coerce')
    z.index = pd.DatetimeIndex(pd.to_datetime(number, unit=unit, utc=True), name='open_time')
    return z[~z.index.duplicated(keep='last')].sort_index().astype('float64')


def load_archives(manifest, folder):
    parts = [parse_kline_zip(Path(folder) / f"{r.symbol}-1m-{r.month}.zip") for r in manifest.itertuples() if r.status != 'not_listed']
    raw = pd.concat(parts).sort_index(kind='stable')
    return raw[~raw.index.duplicated(keep='last')]


def regularize(raw, start=None, end=None):
    """Notebook 01 gap policy: complete UTC minute grid; missing runs of <=5 minutes get forward-filled OHLC and
    zero volume; longer gaps stay NaN."""
    full = pd.date_range(raw.index.min() if start is None else start, raw.index.max() if end is None else end,
                         freq='min', tz='UTC')
    out = raw.reindex(full); out.index.name = 'open_time'
    missing = ~full.isin(raw.index)
    if missing.any():
        changes = np.r_[True, missing[1:] != missing[:-1]]
        grp = np.cumsum(changes) - 1
        runs = np.bincount(grp[missing], minlength=int(grp.max()) + 1)
        short = missing & (runs[grp] <= 5)
        for c in ['open', 'high', 'low', 'close']:
            out.loc[short, c] = out[c].ffill().loc[short]
        for c in ['volume', 'taker_buy_vol']:
            out.loc[short, c] = 0.0
    return out.astype('float64')


# --- chunked_hs, as in notebook 01 ---
def chunked_hs(close,M=32,K=8,chunk=100_000):
    p=close.to_numpy(dtype=np.float32)
    result=np.full(len(p),np.nan,dtype=np.float64)
    for start in range(M-1,len(p),chunk):
        end=min(len(p),start+chunk)
        w=sliding_window_view(p[start-M+1:end],M)
        zero_mean=w-w.mean(axis=1,keepdims=True)
        result[start:end]=np.abs(np.fft.rfft(zero_mean,axis=1))[:,-K:].mean(axis=1)
        if start==(M-1) or end==len(p): print(f'FFT computation {end:,}/{len(p):,}',flush=True)
    return pd.Series(result,index=close.index)



def compute_spot_features(px, with_hs=True):
    """Exactly the notebook 01 signal formulas (z_hs, z_vol, tsr), float32 like the cache."""
    close = px['close']
    r = np.log(close).diff()
    M = 32; rolling_min = 30 * 1440
    out = {}
    if with_hs:
        hs = chunked_hs(close, M=32, K=8)
        log_hs = np.log1p(hs)
        hs_roll = log_hs.rolling(rolling_min, min_periods=rolling_min // 2)
        out['z_hs'] = ((log_hs - hs_roll.mean().shift(1)) / hs_roll.std().shift(1).replace(0, np.nan)).astype('float32')
    rv = r.rolling(M - 1, min_periods=M - 1).std()
    lrv = np.log(rv.where(rv > 0))
    rv_base = lrv.rolling(rolling_min, min_periods=rolling_min // 2)
    out['z_vol'] = ((lrv - rv_base.mean().shift(1)) / rv_base.std().shift(1).replace(0, np.nan)).astype('float32')
    out['tsr'] = ((px.volume - px.taker_buy_vol) / px.volume.replace(0, np.nan)).clip(0, 1).astype('float32')
    return pd.DataFrame(out, index=px.index)


# --- price_only_catalog, as in notebook 07 ---
def price_only_catalog(close,volume,index,H=240,decline=.04,sustain=5,refractory=1440):
    c=np.asarray(close,dtype=np.float64)
    obs=np.isfinite(c)&(c>0)&np.isfinite(volume)&(np.asarray(volume)>0)
    prior=pd.Series(np.where(obs,c,np.nan)).rolling(H,min_periods=H).max().shift(1).to_numpy()
    raw=obs & np.isfinite(prior) & (c<=(1-decline)*prior)
    fresh=raw & ~np.r_[False,raw[:-1]]
    rows=[];last_confirm=-10**12
    for i in np.flatnonzero(fresh):
        if i+sustain>len(c) or i-last_confirm<refractory:continue
        threshold=(1-decline)*prior[i]
        if not(obs[i:i+sustain].all() and np.all(c[i:i+sustain]<=threshold)):continue
        if i-H<0 or not obs[i-H:i].all():continue
        prev=c[i-H:i];peak_i=int(i-H+np.argmax(prev))
        if peak_i<=last_confirm:continue
        onset_candidates=np.flatnonzero(c[peak_i+1:i+1]<=.99*c[peak_i])
        if not len(onset_candidates):continue
        oi=int(peak_i+1+onset_candidates[0])
        rows.append({'event_id':len(rows)+1,'peak_utc':index[peak_i],
                     'onset_1pct_utc':index[oi], 'breach_utc':index[i],
                     'confirmed_utc':index[i+sustain-1],
                     'peak_close':float(c[peak_i]),'breach_close':float(c[i]),
                     'decline_at_breach_pct':(c[i]/c[peak_i]-1)*100,
                     'peak_to_breach_minutes':i-peak_i,
                     'onset_to_breach_minutes':i-oi})
        last_confirm=i+sustain-1
    df=pd.DataFrame(rows)
    if not len(df):raise RuntimeError('No price-only crash events generated; inspect price history')
    df['year']=pd.DatetimeIndex(df.breach_utc).year
    return df

# Shared with notebook 16 (identical code): VPIN, causal feature set, GBM settings.
REGIME_WINDOW_D = 30; RV_WINDOW = 1440; VPIN_BUCKETS_PER_DAY = 50; VPIN_WINDOW = 50; VPIN_LOOKBACK_D = 30
GBM_PARAMS = dict(max_iter=300, learning_rate=0.05, max_leaf_nodes=31, min_samples_leaf=200, l2_regularization=1.0,
                  class_weight='balanced', early_stopping=False, random_state=17)
FEATS = ['z_vol', 'tsr', 'z_hs', 'ret_5', 'ret_15', 'ret_60', 'ret_240', 'rv_15', 'rv_60', 'rv_240', 'rv_ratio_15_240',
         'dd_240', 'dd_1440', 'vol_intensity_15', 'sell_ratio_15', 'sell_ratio_60', 'vpin']

@njit(cache=False)
def _vpin_core(vol, buy, bucket, n_win):
    out = np.full(len(vol), np.nan); ring = np.zeros(n_win); k = 0; run = 0.0; cv = 0.0; cb = 0.0
    for i in range(len(vol)):
        v = vol[i]; b = buy[i]
        if not (v == v) or not (b == b) or v < 0:
            v = 0.0; b = 0.0
        cv += v; cb += min(max(b, 0.0), v)
        bs = bucket[i]
        if bs == bs and bs > 0 and cv >= bs:
            imb = abs(2.0 * cb - cv) / cv
            j = k % n_win; run += imb - ring[j]; ring[j] = imb; k += 1; cv = 0.0; cb = 0.0
        if k >= n_win:
            out[i] = run / n_win
    return out


def vpin_series(px):
    vol = px.volume.to_numpy(float); buy = px.taker_buy_vol.to_numpy(float)
    daily = px.volume.groupby(px.index.floor('D')).sum()
    size = (daily.rolling(VPIN_LOOKBACK_D, min_periods=VPIN_LOOKBACK_D).mean().shift(1) / VPIN_BUCKETS_PER_DAY)
    bucket = size.reindex(px.index.floor('D')).to_numpy(float)
    return _vpin_core(vol, buy, bucket, VPIN_WINDOW)


def causal_features(px, sg):
    c = px.close.astype(float); lc = np.log(c); r = lc.diff(); vol = px.volume.astype(float); sell = vol - px.taker_buy_vol.astype(float)
    F = pd.DataFrame(index=px.index)
    F['z_vol'] = sg.z_vol.astype(float); F['tsr'] = sg.tsr.astype(float); F['z_hs'] = sg.z_hs.astype(float)
    for k in (5, 15, 60, 240):
        F[f'ret_{k}'] = lc - lc.shift(k)
    for k in (15, 60, 240):
        F[f'rv_{k}'] = r.rolling(k, min_periods=k // 2).std()
    F['rv_ratio_15_240'] = F.rv_15 / F.rv_240
    F['dd_240'] = 1 - c / c.rolling(240, min_periods=120).max()
    F['dd_1440'] = 1 - c / c.rolling(1440, min_periods=720).max()
    F['vol_intensity_15'] = vol.rolling(15, min_periods=8).sum() / (vol.rolling(1440, min_periods=720).sum() / 96)
    F['sell_ratio_15'] = sell.rolling(15, min_periods=8).sum() / vol.rolling(15, min_periods=8).sum()
    F['sell_ratio_60'] = sell.rolling(60, min_periods=30).sum() / vol.rolling(60, min_periods=30).sum()
    F['vpin'] = vpin_series(px)
    return F.replace([np.inf, -np.inf], np.nan).astype('float32')
# Shared by notebooks 18 and 19: exact block-bootstrap AUC decomposition.
def week_pairs(sp, wp, sn, wn):
    """C[k, l] = over positives in block k, the number of negatives in block l with a lower score (+0.5 per tie).
    With block multiplicities w from the bootstrap, the AUC is w'Cw / ((w'P)(w'N)); w = 1 gives the ordinary AUC."""
    C = np.zeros((N_WEEKS, N_WEEKS))
    if len(sp) == 0 or len(sn) == 0:
        return C
    o = np.lexsort((sn, wn)); sn = sn[o]; wn = wn[o]
    bounds = np.searchsorted(wn, np.arange(N_WEEKS + 1), 'left')
    for l in range(N_WEEKS):
        a, b = bounds[l], bounds[l + 1]
        if a == b:
            continue
        seg = sn[a:b]; lo = np.searchsorted(seg, sp, 'left'); hi = np.searchsorted(seg, sp, 'right')
        C[:, l] = np.bincount(wp, weights=lo + 0.5 * (hi - lo), minlength=N_WEEKS)
    return C


def _ratio(a, b):
    return a / b if b > 0 else np.nan


STAT_KEYS = ['auc_standard', 'auc_pre', 'auc_in', 'share_in', 'diff_standard_minus_pre', 'in_share_of_excess', 'auc_calm',
             'auc_standard_within_regime', 'auc_pre_within_regime', 'auc_in_within_regime']


def make_parts(sm, code, calm, wk, dec):
    neg = code == 0; pre = code == 1; inn = code == 2; cnt = lambda k: np.bincount(wk[k], minlength=N_WEEKS).astype(float)
    P = dict(C_pre=week_pairs(sm[pre], wk[pre], sm[neg], wk[neg]), C_in=week_pairs(sm[inn], wk[inn], sm[neg], wk[neg]),
             C_calm=week_pairs(sm[(code > 0) & calm], wk[(code > 0) & calm], sm[neg & calm], wk[neg & calm]),
             P_pre=cnt(pre), P_in=cnt(inn), N=cnt(neg), P_calm=cnt((code > 0) & calm), N_calm=cnt(neg & calm),
             Cs_pre=np.zeros((N_WEEKS, N_WEEKS)), Cs_in=np.zeros((N_WEEKS, N_WEEKS)), Pq_pre=[], Pq_in=[], Nq=[])
    for q in range(10):
        kq = dec == q; nq = neg & kq; pq = pre & kq; iq = inn & kq
        P['Cs_pre'] += week_pairs(sm[pq], wk[pq], sm[nq], wk[nq]); P['Cs_in'] += week_pairs(sm[iq], wk[iq], sm[nq], wk[nq])
        P['Pq_pre'].append(cnt(pq)); P['Pq_in'].append(cnt(iq)); P['Nq'].append(cnt(nq))
    for k in ('Pq_pre', 'Pq_in', 'Nq'):
        P[k] = np.array(P[k])
    return P


def stats_from(P, w):
    q = lambda C: w @ C @ w
    Wn = w @ P['N']; Wp = w @ P['P_pre']; Wi = w @ P['P_in']; sp = q(P['C_pre']); si = q(P['C_in'])
    std = _ratio(sp + si, (Wp + Wi) * Wn); pre = _ratio(sp, Wp * Wn); inn = _ratio(si, Wi * Wn); pi = _ratio(Wi, Wp + Wi)
    nq = P['Nq'] @ w; pq = P['Pq_pre'] @ w; iq = P['Pq_in'] @ w; ssp = q(P['Cs_pre']); ssi = q(P['Cs_in'])
    return dict(auc_standard=std, auc_pre=pre, auc_in=inn, share_in=pi, diff_standard_minus_pre=std - pre,
                in_share_of_excess=_ratio(pi * (inn - 0.5), std - 0.5) if np.isfinite(inn) else 0.0,
                auc_calm=_ratio(q(P['C_calm']), (w @ P['P_calm']) * (w @ P['N_calm'])),
                auc_standard_within_regime=_ratio(ssp + ssi, ((pq + iq) * nq).sum()),
                auc_pre_within_regime=_ratio(ssp, (pq * nq).sum()), auc_in_within_regime=_ratio(ssi, (iq * nq).sum()))
import joblib

OUT = SPOT_BASE / 'peak_window_v1'; OUT.mkdir(parents=True, exist_ok=True)
M16 = SPOT_BASE / 'methods_definitions_v1'
M21 = SPOT_BASE / 'onset_positioning_volatility_v1'
DNAME, DCL = '4pct', 0.04
H_LIST = [120, 240, 480, 1440]; H_REF = 240
BASE_METHODS = ['M0_VF_logit', 'M1_vol_threshold', 'M2_VPIN', 'M3_GBM_rich']
SCORES = BASE_METHODS + ['B0_RV24h_baseline']
N_BOOT = 500; N_NULL = 1000; SEED = 20261105; BLOCK_DAYS = 7
SHORT_BAND = (240, 1440)
ROBUST_LOWER = 0.25
EVAL_LAST = ts(EVAL_END).value - (LEAD_MAX + 5) * NS

for _p in [M16 / 'METHODS_FROZEN.json', M16 / 'METHODS_SPEC.json', M16 / '4_methods_definitions_nulls.csv', M21 / '1_onset_per_minute.csv']:
    if not _p.is_file():
        raise FileNotFoundError(f'Results of notebooks 16 and 21 are required: {_p}')
FROZEN16 = json.loads((M16 / 'METHODS_FROZEN.json').read_text()); SPEC16 = json.loads((M16 / 'METHODS_SPEC.json').read_text())
if sha256_file(M16 / 'METHODS_SPEC.json') != FROZEN16['spec_sha256'] or SPEC16['gbm_params'] != GBM_PARAMS or SPEC16['gbm_features'] != FEATS:
    raise RuntimeError('GUARD FAIL: notebook-16 spec/settings differ')

SPEC = {'title': 'Reviewer check: look-back window of the crash peak (dev 2023-01..2026-04, exploratory)',
        'varied': {'H_minutes': H_LIST, 'reference': H_REF,
                   'rule': 'notebook-2 price_only_catalog(close, volume, index, H=H, decline=0.04); peak = max close in the H minutes before the breach; '
                           'onset = first close after the peak at least 1% below it; sustain 5, refractory 1440 and new-peak rules unchanged'},
        'fixed': {'decline': DCL, 'lead_window_min': [LEAD_MIN, LEAD_MAX], 'scores_and_alerts': 'notebook-16 frozen models and dev alarms (4%), not refitted',
                  'per_minute': 'notebook-18/21 decomposition (eval_case) with labels and in-decline split from each H catalog',
                  'nulls': {'circular': '>= 7 days', 'short': f'+-{SHORT_BAND} minutes', 'regime': '24h-RV decile, +-30 days, >= 240 min away'},
                  'null_draws': 'the same shifted alert sets are scored against every H (common random numbers)'},
        'reading_rule': f'Main quantity: M0 in-decline share of the above-chance per-minute AUC with its {BLOCK_DAYS}-day block-bootstrap 95% interval. '
                        f'The claim that in-progress detection is a substantial part of standard skill is called robust to H if the lower bound is >= {ROBUST_LOWER} '
                        'at every H; otherwise the H at which it fails is reported. Expected mechanical direction: share rises with H.',
        'secondary_reading': 'At an H where the rule fails, the larger pre-onset share is read as timing information only if M0 within-regime pre-onset AUC '
                             'at that H lies above its H=240 value with non-overlapping 95% intervals; otherwise it is attributed to the changed event set '
                             '(shorter falls), not to warning skill. Written before any result on real data.',
        'draws': N_NULL, 'bootstrap': f'{N_BOOT} x {BLOCK_DAYS}-day blocks', 'seed': SEED, 'notebook16_spec_sha256': FROZEN16['spec_sha256'],
        'holdouts': 'not used'}
_raw = (json.dumps(SPEC, indent=2, sort_keys=True, default=str) + '\n').encode()
_sp = OUT / '22_SPEC.json'
if _sp.exists() and _sp.read_bytes() != _raw:
    _sp.rename(OUT / f'22_SPEC.replaced_{pd.Timestamp.now(tz="UTC"):%Y%m%dT%H%M%S}.json')
_sp.write_bytes(_raw); SPEC_SHA = hashlib.sha256(_raw).hexdigest()
print('Output:', OUT, '| spec sha256', SPEC_SHA[:16])

Mounted at /content/drive
SPOT cache verified. Frozen spot thresholds: {'TH_CAL': 0.6698990591069133, 'TH_HIGH': 0.6864782740093911, 'TH_BUCKET': 0.7166067355142407}
Output: /content/drive/MyDrive/btc_lead_cost_results_77months_spot/peak_window_v1 | spec sha256 5819cf8e557ca20e


## 1. Data, scores and alerts (notebook 16 reproduction check)

In [2]:
t_ = time.time()
px, sg = load_btc_spot()
F = causal_features(px, sg)
T = px.index.asi8; close = px.close.to_numpy(float); volume = px.volume.to_numpy(float)
v_ = sg.z_vol.to_numpy(float); f_ = sg.tsr.to_numpy(float); h_ = sg.z_hs.to_numpy(float)
VALID = (np.isfinite(close) & (close > 0) & np.isfinite(volume) & (volume > 0) & np.isfinite(v_) & np.isfinite(f_) & np.isfinite(h_) & (f_ >= 0) & (f_ <= 1))
EVM = VALID & (T >= ts(EVAL_START).value) & (T < EVAL_LAST)
DD240 = F.dd_240.to_numpy(float); CALM = np.isfinite(DD240) & (DD240 < DCL / 4)
_lc = np.log(np.where(np.isfinite(close) & (close > 0), close, np.nan))
RV = pd.Series(_lc).diff().rolling(RV_WINDOW, min_periods=RV_WINDOW // 2).std().to_numpy()
DEC = np.full(len(T), -1, np.int64); _okr = EVM & np.isfinite(RV)
DEC[_okr] = np.minimum((pd.Series(RV[_okr]).rank(pct=True).to_numpy() * 10).astype(int), 9)
WEEK = np.where(EVM, (T - ts(EVAL_START).value) // (BLOCK_DAYS * 1440 * NS), -1).astype(np.int64)
N_WEEKS = int(WEEK[EVM].max()) + 1

TCOLS = ['peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc']


def catalog_H(H):
    E_ = price_only_catalog(close, volume, px.index, H=H, decline=DCL)
    for c_ in TCOLS:
        E_[c_] = pd.to_datetime(E_[c_], utc=True)
    return E_.sort_values('breach_utc').reset_index(drop=True)


E_ref = load_catalog(SPOT_BASE / 'event_definition_eligibility_audit' / 'price_only_event_catalog.csv')
CAT = {H: catalog_H(H) for H in H_LIST}
E240 = CAT[H_REF]
if len(E240) != len(E_ref) or any(not np.array_equal(E240[c_].astype('int64').to_numpy(), E_ref[c_].astype('int64').to_numpy()) for c_ in TCOLS):
    raise RuntimeError('GUARD FAIL: H=240 catalog differs from the notebook-2 catalog')

Xl = F[['z_vol', 'tsr']].to_numpy(float); okl = train_mask(T, VALID) & np.isfinite(Xl).all(axis=1)
cfg0 = FROZEN16['configs'][f'M0_VF_logit|{DNAME}']
m0 = new_logit().fit(Xl[okl], label_from_catalog(T, E240)[okl])
if not (np.allclose(m0[-1].coef_.ravel(), cfg0['coef'], rtol=1e-6, atol=1e-9) and np.isclose(m0[-1].intercept_[0], cfg0['intercept'], rtol=1e-6, atol=1e-9)):
    raise RuntimeError('GUARD FAIL: M0 refit differs from notebook 16')
cfg3 = FROZEN16['configs'][f'M3_GBM_rich|{DNAME}']; mp = SPOT_BASE / cfg3['model_file']
if sha256_file(mp) != cfg3['model_sha256']:
    raise RuntimeError('GUARD FAIL: notebook-16 GBM file SHA-256 differs')
g3 = joblib.load(mp)
IDX = np.flatnonzero(EVM); S = {}
s = np.full(len(T), np.nan); okp = EVM & np.isfinite(Xl).all(axis=1); s[okp] = m0.predict_proba(Xl[okp])[:, 1]; S['M0_VF_logit'] = s
S['M1_vol_threshold'] = np.where(VALID, F.z_vol.to_numpy(float), np.nan)
S['M2_VPIN'] = np.where(VALID, F.vpin.to_numpy(float), np.nan)
s = np.full(len(T), np.nan); s[IDX] = g3.predict_proba(F[FEATS].to_numpy(np.float32)[IDX])[:, 1]; S['M3_GBM_rich'] = s
S['B0_RV24h_baseline'] = np.where(VALID, RV, np.nan)

ALERTS = {}
for mname in BASE_METHODS:
    a = np.sort(pd.to_datetime(pd.read_csv(M16 / f'alarms_{mname}_{DNAME}.csv').alarm_utc, utc=True).astype('int64').to_numpy())
    if len(a) != FROZEN16['configs'][f'{mname}|{DNAME}']['dev_alerts']:
        raise RuntimeError(f'GUARD FAIL: notebook-16 alarms for {mname} changed')
    ALERTS[mname] = a
print(f'catalogs built for H = {H_LIST}; H=240 catalog, M0 coefficients, GBM file and alarms verified ({time.time() - t_:.0f}s)')

catalogs built for H = [120, 240, 480, 1440]; H=240 catalog, M0 coefficients, GBM file and alarms verified (74s)


## 2. Event catalogs by H

- `onset_ge_240_before_breach`: share of events whose onset is at least 240 minutes before the breach; the whole judging window (30–240 minutes before) is then already in decline, so pre-onset warning is impossible by definition.
- `onset_lt_30_before_breach`: share of events whose onset is less than 30 minutes before the breach; for these, in-progress detection is impossible.
- `shared_with_H240`: number of events with the same breach time as an H = 240 event.

In [3]:
EVS = {H: events_between(CAT[H], EVAL_START, EVAL_END) for H in H_LIST}
br240 = set(EVS[H_REF].breach_utc.astype('int64'))
desc = []
for H in H_LIST:
    ev = EVS[H]; gap_on = (ev.breach_utc - ev.onset_1pct_utc).dt.total_seconds() / 60; gap_pk = (ev.breach_utc - ev.peak_utc).dt.total_seconds() / 60
    desc.append(dict(H=H, catalog_events=len(CAT[H]), dev_events=len(ev), shared_with_H240=int(ev.breach_utc.astype('int64').isin(br240).sum()),
                     median_peak_to_breach_min=float(gap_pk.median()), median_onset_to_breach_min=float(gap_on.median()),
                     p90_onset_to_breach_min=float(gap_on.quantile(0.9)),
                     onset_ge_240_before_breach=float((gap_on >= LEAD_MAX).mean()), onset_lt_30_before_breach=float((gap_on < LEAD_MIN).mean())))
    ev[['event_id', 'peak_utc', 'onset_1pct_utc', 'breach_utc', 'confirmed_utc', 'peak_to_breach_minutes', 'onset_to_breach_minutes']].to_csv(
        OUT / f'1_events_H{H}.csv', index=False)          # timestamps only, no prices
DESC = pd.DataFrame(desc); DESC.to_csv(OUT / '1_catalog_by_window.csv', index=False); display(DESC.round(3))

,H,catalog_events,dev_events,shared_with_H240,median_peak_to_breach_min,median_onset_to_breach_min,p90_onset_to_breach_min,onset_ge_240_before_breach,onset_lt_30_before_breach
0,120,166,51,27,102.0,54.0,96.0,0.000,0.255
1,240,269,83,83,178.0,119.0,182.8,0.000,0.096
2,480,394,129,48,351.0,240.0,427.2,0.504,0.054
3,1440,562,218,29,997.0,819.5,1310.3,0.862,0.018


## 3. Per-minute AUC decomposition by H

For each H, I rebuild the labels (positive = 30–240 minutes before a breach) and the in-progress split from that H's catalog and re-evaluate the same scores.

In [4]:
def eval_case(s, y, inn, base, seed, guard=False):
    m = base & np.isfinite(s) & (DEC >= 0)
    code = np.where(y[m] == 1, np.where(inn[m], 2, 1), 0).astype(np.int64)
    P = make_parts(s[m], code, CALM[m], WEEK[m], DEC[m])
    point = stats_from(P, np.ones(N_WEEKS))
    point.update(n_pre=int((code == 1).sum()), n_in=int((code == 2).sum()), n_neg=int((code == 0).sum()))
    if guard:
        from sklearn.metrics import roc_auc_score
        keep = code != 2
        if abs(roc_auc_score((code[keep] > 0).astype(int), s[m][keep]) - point['auc_pre']) > 1e-9:
            raise RuntimeError('GUARD FAIL: block-pair AUC differs from sklearn')
        print('PASS: block-pair AUC equals sklearn')
    rng = np.random.default_rng(seed); boot = {k: np.empty(N_BOOT) for k in STAT_KEYS}
    for i in range(N_BOOT):
        r = stats_from(P, np.bincount(rng.integers(0, N_WEEKS, N_WEEKS), minlength=N_WEEKS).astype(float))
        for k in STAT_KEYS:
            boot[k][i] = r[k]
    for k in STAT_KEYS:
        lo, hi = np.nanpercentile(boot[k], [2.5, 97.5]) if np.isfinite(boot[k]).any() else (np.nan, np.nan)
        point[f'{k}_lo'] = float(lo); point[f'{k}_hi'] = float(hi)
    return point


def in_decline_mask(E_, y_):
    """Positive minutes at or after the onset of the event that makes them positive."""
    br = E_.breach_utc.astype('int64').to_numpy(); on = E_.onset_1pct_utc.astype('int64').to_numpy()
    pos = np.searchsorted(br, T + LEAD_MIN * NS, 'left'); ok = pos < len(br)
    inn = np.zeros(len(T), bool); inn[ok] = T[ok] >= on[pos[ok]]
    return inn & (y_ == 1)


R21 = pd.read_csv(M21 / '1_onset_per_minute.csv')
prow = []; ci = 0; t_ = time.time()
for H in H_LIST:
    yH = label_from_catalog(T, CAT[H]); innH = in_decline_mask(CAT[H], yH)
    for sname in SCORES:
        r = eval_case(S[sname], yH, innH, EVM, SEED + 5000 + ci, guard=(ci == 0)); ci += 1
        prow.append(dict(H=H, score=sname, **r))
        if H == H_REF:
            ref = R21[(R21.onset == '1pct') & (R21.score == sname)]
            if len(ref) != 1 or any(abs(r[k] - float(ref.iloc[0][k])) > 1e-9 for k in ['auc_standard', 'auc_pre', 'in_share_of_excess']):
                raise RuntimeError(f'GUARD FAIL: H=240 per-minute result for {sname} differs from notebook 21')
    print(f'H={H}: per-minute decomposition done ({time.time() - t_:.0f}s)', flush=True)
print('PASS: H=240 per-minute results equal notebook 21')
PM = pd.DataFrame(prow); PM.to_csv(OUT / '2_per_minute_by_window.csv', index=False)
display(PM.pivot(index='score', columns='H', values='auc_standard').round(3))
display(PM.pivot(index='score', columns='H', values='auc_pre').round(3))
display(PM.pivot(index='score', columns='H', values='in_share_of_excess').round(3))
display(PM.pivot(index='score', columns='H', values='auc_pre_within_regime').round(3))

PASS: block-pair AUC equals sklearn
H=120: per-minute decomposition done (14s)
H=240: per-minute decomposition done (29s)
H=480: per-minute decomposition done (47s)
H=1440: per-minute decomposition done (69s)
PASS: H=240 per-minute results equal notebook 21


H,120,240,480,1440
score,,,,
B0_RV24h_baseline,0.748,0.748,0.714,0.677
M0_VF_logit,0.734,0.763,0.762,0.696
M1_vol_threshold,0.735,0.764,0.763,0.696
M2_VPIN,0.410,0.456,0.470,0.495
M3_GBM_rich,0.719,0.755,0.785,0.753


H,120,240,480,1440
score,,,,
B0_RV24h_baseline,0.737,0.709,0.670,0.693
M0_VF_logit,0.699,0.673,0.647,0.707
M1_vol_threshold,0.700,0.674,0.648,0.708
M2_VPIN,0.406,0.436,0.416,0.390
M3_GBM_rich,0.683,0.654,0.611,0.617


H,120,240,480,1440
score,,,,
B0_RV24h_baseline,0.180,0.505,0.767,0.906
M0_VF_logit,0.268,0.615,0.835,0.909
M1_vol_threshold,0.266,0.613,0.834,0.909
M2_VPIN,NaN,NaN,NaN,NaN
M3_GBM_rich,0.283,0.646,0.886,0.960


H,120,240,480,1440
score,,,,
B0_RV24h_baseline,0.495,0.511,0.511,0.505
M0_VF_logit,0.606,0.592,0.584,0.647
M1_vol_threshold,0.607,0.594,0.585,0.648
M2_VPIN,0.436,0.464,0.433,0.407
M3_GBM_rich,0.566,0.547,0.521,0.508


## 4. Alert level: pre-onset warning and in-progress detection by H (same alerts)

I keep the alerts frozen in notebook 16; only their matching to events and the onset times of those events change with H. For the nulls I score the same shifted alert sets against every H instead of drawing new sets for each H.

In [ ]:
def make_draw_space(start, end):
    okd = in_period(T, start, end) & np.isfinite(RV)
    tt = T[okd]; dq = np.minimum((pd.Series(RV[okd]).rank(pct=True).to_numpy() * 10).astype(int), 9)
    o = np.lexsort((tt, dq)); ds = dq[o]
    return dict(t0=ts(start).value, L=(ts(end).value - ts(start).value) // NS, TT=tt, DD=dq, PP=tt[o],
                STARTS=np.searchsorted(ds, np.arange(10), 'left'), ENDS=np.searchsorted(ds, np.arange(10), 'right'))


def draw_circular(a, rng, sp):
    d = int(rng.integers(7 * 1440, sp['L'] - 7 * 1440))
    return np.sort(sp['t0'] + (a - sp['t0'] + d * NS) % (sp['L'] * NS))


def draw_short(a, rng, sp):
    d = int(rng.integers(SHORT_BAND[0], SHORT_BAND[1] + 1)) * (1 if rng.random() < 0.5 else -1)
    return np.sort(sp['t0'] + (a - sp['t0'] + d * NS) % (sp['L'] * NS))


def draw_regime(a, rng, sp, W=30 * 1440 * NS):
    TT_, PP_, STARTS_, ENDS_ = sp['TT'], sp['PP'], sp['STARTS'], sp['ENDS']
    j = np.clip(np.searchsorted(TT_, a, 'right') - 1, 0, len(TT_) - 1); ad = sp['DD'][j]
    lo = np.array([STARTS_[q] + np.searchsorted(PP_[STARTS_[q]:ENDS_[q]], ti - W, 'left') for ti, q in zip(a, ad)])
    hi = np.array([STARTS_[q] + np.searchsorted(PP_[STARTS_[q]:ENDS_[q]], ti + W, 'right') for ti, q in zip(a, ad)])
    c = PP_[lo + (rng.random(len(a)) * (hi - lo)).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
    for _ in range(20):
        if not bad.any():
            break
        c[bad] = PP_[lo[bad] + (rng.random(int(bad.sum())) * (hi[bad] - lo[bad])).astype(np.int64)]; bad = np.abs(c - a) < LEAD_MAX * NS
    return np.sort(c)


NULLS = [('circular >=7d', draw_circular), ('short 4h-1d', draw_short), ('regime-matched', draw_regime)]
DSP = make_draw_space(EVAL_START, EVAL_END)


def counts(a, ev):
    m = match_events(ev, np.sort(np.asarray(a, np.int64)))
    pre = int(m.pre1pct.sum()) if len(m) else 0
    return pre, len(m) - pre, len(m)


R16 = pd.read_csv(M16 / '4_methods_definitions_nulls.csv')
rows = []; t_ = time.time()
for mi, mname in enumerate(BASE_METHODS):
    a = ALERTS[mname]; obs = {H: counts(a, EVS[H]) for H in H_LIST}
    r16 = R16[(R16.method == mname) & (R16.definition == DNAME)].iloc[0]
    if obs[H_REF] != (int(r16.pre), int(r16.post), int(r16.caught)):
        raise RuntimeError(f'GUARD FAIL: {mname} counts at H=240 differ from notebook 16')
    for ni, (nname, drawer) in enumerate(NULLS):
        rng = np.random.default_rng(SEED + 100 * mi + ni)
        pre_d = {H: np.empty(N_NULL) for H in H_LIST}; post_d = {H: np.empty(N_NULL) for H in H_LIST}
        for i in range(N_NULL):
            sh = drawer(a, rng, DSP)
            for H in H_LIST:
                pre_d[H][i], post_d[H][i], _ = counts(sh, EVS[H])
        for H in H_LIST:
            po, qo, caught = obs[H]
            rows.append(dict(H=H, method=mname, null=nname, alerts=len(a), events=len(EVS[H]), caught=caught, pre=po, post=qo,
                             pre_null_mean=float(pre_d[H].mean()), post_null_mean=float(post_d[H].mean()),
                             p_pre=(1 + int((pre_d[H] >= po).sum())) / (N_NULL + 1), p_post=(1 + int((post_d[H] >= qo).sum())) / (N_NULL + 1),
                             pre_excess_per_100_alerts=100 * (po - float(pre_d[H].mean())) / len(a),
                             post_excess_per_100_alerts=100 * (qo - float(post_d[H].mean())) / len(a)))
    print(f'{mname}: nulls done for all H ({time.time() - t_:.0f}s)', flush=True)
print('PASS: H=240 alert counts equal notebook 16')
AL = pd.DataFrame(rows); AL.to_csv(OUT / '3_alerts_by_window.csv', index=False)
display(AL.pivot_table(index=['method', 'H'], columns='null', values=['pre_excess_per_100_alerts', 'post_excess_per_100_alerts'], sort=False).round(2))
display(AL.pivot_table(index=['method', 'H'], columns='null', values=['p_pre', 'p_post'], sort=False).round(4))

M0_VF_logit: nulls done for all H (54s)
M1_vol_threshold: nulls done for all H (110s)
M2_VPIN: nulls done for all H (169s)


## 5. Figures and summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
g = PM[PM.score == 'M0_VF_logit'].set_index('H').loc[H_LIST]; x = np.arange(len(H_LIST))
axes[0].errorbar(x, g.in_share_of_excess, yerr=[g.in_share_of_excess - g.in_share_of_excess_lo, g.in_share_of_excess_hi - g.in_share_of_excess],
                 fmt='o', color='#2471a3', capsize=4)
axes[0].axhline(ROBUST_LOWER, color='#c0392b', lw=0.8, ls='--', label=f'pre-set lower-bound line ({ROBUST_LOWER})')
axes[0].set_xticks(x); axes[0].set_xticklabels([f'{H} min' for H in H_LIST]); axes[0].set_ylim(0, 1)
axes[0].set_title('M0: in-decline share of the above-chance per-minute AUC\nby look-back window of the crash peak (95% block-bootstrap CI)', fontsize=9)
axes[0].legend(fontsize=8)
g = AL[(AL.null == 'regime-matched') & (AL.method == 'M0_VF_logit')].set_index('H').loc[H_LIST]
axes[1].bar(x - 0.2, g.pre_excess_per_100_alerts, 0.4, color='#2471a3', label='pre-onset warning')
axes[1].bar(x + 0.2, g.post_excess_per_100_alerts, 0.4, color='#95a5a6', label='in-progress detection')
axes[1].axhline(0, color='k', lw=0.8); axes[1].set_xticks(x); axes[1].set_xticklabels([f'{H} min' for H in H_LIST])
axes[1].set_title('M0: above-chance catches per 100 alerts (regime-matched null)', fontsize=9); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / '4_peak_window.png', dpi=140); plt.show()

m0pm = PM[PM.score == 'M0_VF_logit'].set_index('H')
robust = bool((m0pm.loc[H_LIST, 'in_share_of_excess_lo'] >= ROBUST_LOWER).all())
fails = [int(H) for H in H_LIST if not m0pm.loc[H, 'in_share_of_excess_lo'] >= ROBUST_LOWER]
secondary = {int(H): ('timing information' if m0pm.loc[H, 'auc_pre_within_regime_lo'] > m0pm.loc[H_REF, 'auc_pre_within_regime_hi']
                      else 'changed event set, not warning skill') for H in fails}
summary = dict(spec_sha256=SPEC_SHA, status='exploratory; dev 2023-01..2026-04 only', reading_rule=SPEC['reading_rule'],
               secondary_reading=SPEC['secondary_reading'], robust_to_H=robust, H_failing_rule=fails, secondary_verdict=secondary,
               catalogs=DESC.to_dict('records'),
               per_minute=PM[['H', 'score', 'auc_standard', 'auc_pre', 'auc_in', 'share_in', 'in_share_of_excess', 'in_share_of_excess_lo', 'in_share_of_excess_hi',
                              'auc_pre_within_regime', 'auc_pre_within_regime_lo', 'auc_pre_within_regime_hi', 'n_pre', 'n_in']].to_dict('records'),
               alerts=AL[['H', 'method', 'null', 'alerts', 'events', 'caught', 'pre', 'post', 'pre_null_mean', 'post_null_mean', 'p_pre', 'p_post',
                          'pre_excess_per_100_alerts', 'post_excess_per_100_alerts']].to_dict('records'))
(OUT / 'PEAK_WINDOW_SUMMARY.json').write_text(json.dumps(summary, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o)))

print(f"Pre-set rule (M0 share lower bound >= {ROBUST_LOWER} at every H): {'MET' if robust else 'NOT MET at H = ' + str(fails)}")
for H, v in secondary.items():
    print(f"Secondary reading at H={H}: within-regime pre-onset AUC {m0pm.loc[H, 'auc_pre_within_regime']:.3f} "
          f"[{m0pm.loc[H, 'auc_pre_within_regime_lo']:.3f}, {m0pm.loc[H, 'auc_pre_within_regime_hi']:.3f}] vs H=240 "
          f"{m0pm.loc[H_REF, 'auc_pre_within_regime']:.3f} [{m0pm.loc[H_REF, 'auc_pre_within_regime_lo']:.3f}, {m0pm.loc[H_REF, 'auc_pre_within_regime_hi']:.3f}] -> {v}")
for H in H_LIST:
    p = m0pm.loc[H]; d = DESC.set_index('H').loc[H]
    a = AL[(AL.H == H) & (AL.method == 'M0_VF_logit') & (AL.null == 'regime-matched')].iloc[0]
    print(f"H={H:5d} | dev events {int(d.dev_events):3d} (shared with 240: {int(d.shared_with_H240):3d}) | median onset->breach {d.median_onset_to_breach_min:5.0f} min "
          f"| M0 AUC std {p.auc_standard:.3f}, pre {p.auc_pre:.3f}, pre within regime {p.auc_pre_within_regime:.3f} "
          f"| in-decline share {p.in_share_of_excess:.2f} [{p.in_share_of_excess_lo:.2f}, {p.in_share_of_excess_hi:.2f}] "
          f"| per 100 alerts: pre {a.pre_excess_per_100_alerts:+.2f} (p {a.p_pre:.3f}), in-progress {a.post_excess_per_100_alerts:+.2f} (p {a.p_post:.3f})")

## Interpretation
- The main result is the "Pre-set rule" line of the final output (`robust_to_H` and `H_failing_rule` in `PEAK_WINDOW_SUMMARY.json`); MET means the four-hour choice does not drive the conclusion.
- A larger in-progress share at longer H follows from the definition (slower declines enter and the onset moves earlier), so the share that remains at H = 120 is the strictest check. With a two-hour peak a 4% fall must complete within two hours, so most of the four-hour judging window lies before onset. At an H where the rule fails, I read a larger pre-onset share as timing information only if M0's within-regime pre-onset AUC lies above its H = 240 value with non-overlapping 95% intervals, and otherwise as a change in the event set (only short falls remain).
- For pre-onset warning, if the within-regime pre-onset AUC (`auc_pre_within_regime`) and the pre-onset excess per 100 alerts change little with H, the conclusion that timing information is weak does not depend on H either.
- Changing H changes the events themselves (`shared_with_H240`), so each H measures the same phenomenon with a different ruler rather than recounting the same events.